# Day 038 — Exercise 1: distribution_summary

**What you'll build:** `distribution_summary(df, col) -> dict` — return a profile of one column: for numeric columns, compute quantiles and descriptive stats; for string columns, compute unique count, top value, and frequency.

**Why it matters:** The first step of every EDA is asking 'what does this column look like?' Automating the answer into a structured dict makes it loggable, testable, and composable into a full report.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd

# SALES_DF: 8 rows, 6 columns
# product:  Widget×4, Gadget×2, Doohickey×2
# revenue   = price × quantity  (pre-computed)
SALES_DF = pd.DataFrame({
    'product':  ['Widget', 'Widget', 'Widget', 'Widget',
                 'Gadget', 'Gadget', 'Doohickey', 'Doohickey'],
    'category': ['Elec', 'Elec', 'Elec', 'Elec',
                 'Elec', 'Elec', 'Access', 'Access'],
    'region':   ['North', 'South', 'East', 'West',
                 'North', 'East', 'North', 'South'],
    'price':    [25.0, 25.0, 25.0, 25.0, 150.0, 150.0, 8.0, 8.0],
    'quantity': [10, 5, 4, 6, 3, 7, 50, 15],
    'revenue':  [250.0, 125.0, 100.0, 150.0, 450.0, 1050.0, 400.0, 120.0],
})

## Your Implementation

In [ ]:
def distribution_summary(df: pd.DataFrame, col: str) -> dict:
    """
    Profile one column.

    Numeric columns → {count, mean, std, min, q25, median, q75, max, null_count}
    String columns  → {count, unique, top, top_freq, null_count}
    """
    s = df[col]
    if pd.api.types.is_numeric_dtype(s):
        # TODO: return dict with count, mean, std, min,
        #       q25 (quantile 0.25), median (quantile 0.50),
        #       q75 (quantile 0.75), max, null_count
        pass
    # TODO: counts = s.value_counts()
    # TODO: return dict with count, unique (nunique()),
    #       top (most frequent value), top_freq, null_count
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined, returns dict
    try:
        assert 'distribution_summary' in globals()
        result = distribution_summary(SALES_DF, 'revenue')
        assert isinstance(result, dict), \
            f'expected dict, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 1: returns a dict')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: numeric column has all required keys
    try:
        r = distribution_summary(SALES_DF, 'revenue')
        for k in ('count', 'mean', 'std', 'min', 'q25', 'median', 'q75', 'max', 'null_count'):
            assert k in r, f'numeric result missing key: {k}'
        passed += 1; print('\u2705 Check 2: numeric result has all 9 keys')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: numeric values are correct
    # revenue: [250,125,100,150,450,1050,400,120] → count=8, min=100, max=1050
    try:
        r = distribution_summary(SALES_DF, 'revenue')
        assert r['count']      == 8,    f'count={r["count"]}, expected 8'
        assert r['min']        == 100.0, f'min={r["min"]}, expected 100'
        assert r['max']        == 1050.0, f'max={r["max"]}, expected 1050'
        assert r['null_count'] == 0,    f'null_count={r["null_count"]}, expected 0'
        passed += 1; print('\u2705 Check 3: count=8, min=100, max=1050, null_count=0')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: quantile ordering q25 <= median <= q75
    try:
        r = distribution_summary(SALES_DF, 'revenue')
        assert r['q25'] <= r['median'] <= r['q75'], \
            f'quantile order violated: q25={r["q25"]}, median={r["median"]}, q75={r["q75"]}'
        passed += 1; print(f'\u2705 Check 4: q25={r["q25"]:.1f} <= median={r["median"]:.1f} <= q75={r["q75"]:.1f}')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: categorical column returns correct keys + values
    # product: Widget×4, Gadget×2, Doohickey×2 → unique=3, top='Widget'
    try:
        r = distribution_summary(SALES_DF, 'product')
        for k in ('count', 'unique', 'top', 'top_freq', 'null_count'):
            assert k in r, f'categorical result missing key: {k}'
        assert r['count']    == 8,       f'count={r["count"]}'
        assert r['unique']   == 3,       f'unique={r["unique"]}, expected 3'
        assert r['top']      == 'Widget', f'top={r["top"]!r}, expected Widget'
        assert r['top_freq'] == 4,       f'top_freq={r["top_freq"]}, expected 4'
        passed += 1; print("\u2705 Check 5: categorical: unique=3, top='Widget', top_freq=4")
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import pandas as pd

def distribution_summary(df: pd.DataFrame, col: str) -> dict:
    s = df[col]
    if pd.api.types.is_numeric_dtype(s):
        return {
            'count':      int(s.count()),
            'mean':       round(float(s.mean()), 4),
            'std':        round(float(s.std()), 4),
            'min':        float(s.min()),
            'q25':        float(s.quantile(0.25)),
            'median':     float(s.quantile(0.50)),
            'q75':        float(s.quantile(0.75)),
            'max':        float(s.max()),
            'null_count': int(s.isnull().sum()),
        }
    counts = s.value_counts()
    return {
        'count':      int(s.count()),
        'unique':     int(s.nunique()),
        'top':        str(counts.index[0]) if len(counts) else None,
        'top_freq':   int(counts.iloc[0])  if len(counts) else 0,
        'null_count': int(s.isnull().sum()),
    }
```

</details>